# 1. Load data

In [1]:
import pandas as pd

data = pd.read_csv("../data/raw/host_countries.csv")
data.head()

,key_id,tournament_id,tournament_name,team_id,team_name,team_code,performance
0,1,WC-1930,1930 FIFA Men's World Cup,T-84,Uruguay,URY,champions
1,2,WC-1934,1934 FIFA Men's World Cup,T-41,Italy,ITA,champions
2,3,WC-1938,1938 FIFA Men's World Cup,T-30,France,FRA,quater-finals
3,4,WC-1950,1950 FIFA Men's World Cup,T-09,Brazil,BRA,runners-up
4,5,WC-1954,1954 FIFA Men's World Cup,T-75,Switzerland,CHE,quarter-final


# 2. Data analysis

In [2]:
data.columns

Index(['key_id', 'tournament_id', 'tournament_name', 'team_id', 'team_name',
       'team_code', 'performance'],
      dtype='str')

In [3]:
data.isna().sum()

key_id             0
tournament_id      0
tournament_name    0
team_id            0
team_name          0
team_code          0
performance        0
dtype: int64

In [4]:
data.duplicated().sum()

np.int64(0)

# 3. Extract year

In [5]:
def extract_year(row: pd.DataFrame) -> str:
    return row["tournament_id"][3:]

years = data.apply(extract_year, axis=1, result_type="expand")
data["Year"] = years
data[["tournament_id", "Year"]]

,tournament_id,Year
0,WC-1930,1930
1,WC-1934,1934
2,WC-1938,1938
3,WC-1950,1950
4,WC-1954,1954
5,WC-1958,1958
6,WC-1962,1962
7,WC-1966,1966
8,WC-1970,1970
9,WC-1974,1974


# 4. Cleaning

## Remove female world cups

In [6]:
data = data[~data["tournament_name"].str.contains("Women")].copy()
data

,key_id,tournament_id,tournament_name,team_id,team_name,team_code,performance,Year
0,1,WC-1930,1930 FIFA Men's World Cup,T-84,Uruguay,URY,champions,1930
1,2,WC-1934,1934 FIFA Men's World Cup,T-41,Italy,ITA,champions,1934
2,3,WC-1938,1938 FIFA Men's World Cup,T-30,France,FRA,quater-finals,1938
3,4,WC-1950,1950 FIFA Men's World Cup,T-09,Brazil,BRA,runners-up,1950
4,5,WC-1954,1954 FIFA Men's World Cup,T-75,Switzerland,CHE,quarter-final,1954
5,6,WC-1958,1958 FIFA Men's World Cup,T-74,Sweden,SWE,runners-up,1958
6,7,WC-1962,1962 FIFA Men's World Cup,T-13,Chile,CHL,third place,1962
7,8,WC-1966,1966 FIFA Men's World Cup,T-28,England,ENG,champions,1966
8,9,WC-1970,1970 FIFA Men's World Cup,T-46,Mexico,MEX,quater-finals,1970
9,10,WC-1974,1974 FIFA Men's World Cup,T-86,West Germany,DEU,champions,1974


## Remove unnecessary columns

In [7]:
data = data.drop(columns=["key_id", "tournament_id", "tournament_name", "team_id", "team_code"])
data.columns

Index(['team_name', 'performance', 'Year'], dtype='str')

## Columns rename

In [8]:
columns = {
    "team_name": "Host",
    "performance": "Performance"
}

data = data.rename(columns=columns)
data

,Host,Performance,Year
0,Uruguay,champions,1930
1,Italy,champions,1934
2,France,quater-finals,1938
3,Brazil,runners-up,1950
4,Switzerland,quarter-final,1954
5,Sweden,runners-up,1958
6,Chile,third place,1962
7,England,champions,1966
8,Mexico,quater-finals,1970
9,West Germany,champions,1974


## Convert values

In [9]:
data["Host"].value_counts()

Host
Italy            2
France           2
Brazil           2
Mexico           2
Uruguay          1
Switzerland      1
Sweden           1
Chile            1
England          1
West Germany     1
Argentina        1
Spain            1
United States    1
Japan            1
South Korea      1
Germany          1
South Africa     1
Russia           1
Qatar            1
Name: count, dtype: int64

In [10]:
data.loc[data["Host"] == "West Germany", "Host"] = "Germany"
data.loc[data["Host"] == "England", "Host"] = "United Kingdom"
data["Host"].value_counts()

Host
Italy             2
France            2
Brazil            2
Mexico            2
Germany           2
Uruguay           1
Switzerland       1
Sweden            1
Chile             1
United Kingdom    1
Argentina         1
Spain             1
United States     1
Japan             1
South Korea       1
South Africa      1
Russia            1
Qatar             1
Name: count, dtype: int64

In [11]:
data["Performance"].value_counts()

Performance
champions             6
quater-finals         4
third place           3
runners-up            2
round of 16           2
fourth place          2
group stage           2
quarter-final         1
second group stage    1
Name: count, dtype: int64

In [12]:
performance = {
    "champions": "First place",
    "round of 16": "Round of 16",
    "quater-finals": "Quarter-final",
    "quarter-final": "Quarter-final",
    "third place": "Third place",
    "runners-up": "Second place",
    "fourth place": "Fourth place",
    "group stage": "Group stage",
    "second group stage": "Second group stage"
}

data["Performance"] = data["Performance"].map(performance)
data

,Host,Performance,Year
0,Uruguay,First place,1930
1,Italy,First place,1934
2,France,Quarter-final,1938
3,Brazil,Second place,1950
4,Switzerland,Quarter-final,1954
5,Sweden,Second place,1958
6,Chile,Third place,1962
7,United Kingdom,First place,1966
8,Mexico,Quarter-final,1970
9,Germany,First place,1974


# 5. World Cup 2026

In [13]:
usa = {
    "Host": "United States",
    "Performance": "Round of 16",
    "Year": 2026
}
mexico = {
    "Host": "Mexico",
    "Performance": "Round of 16",
    "Year": 2026
}
canada = {
    "Host": "Canada",
    "Performance": "Round of 16",
    "Year": 2026
}

wc_2026 = pd.DataFrame([usa, mexico, canada])
wc_2026

,Host,Performance,Year
0,United States,Round of 16,2026
1,Mexico,Round of 16,2026
2,Canada,Round of 16,2026


In [14]:
data = pd.concat([data, wc_2026], ignore_index=True)
data

,Host,Performance,Year
0,Uruguay,First place,1930
1,Italy,First place,1934
2,France,Quarter-final,1938
3,Brazil,Second place,1950
4,Switzerland,Quarter-final,1954
5,Sweden,Second place,1958
6,Chile,Third place,1962
7,United Kingdom,First place,1966
8,Mexico,Quarter-final,1970
9,Germany,First place,1974


# 6. Continents

In [15]:
data["Host"].unique()

<StringArray>
[       'Uruguay',          'Italy',         'France',         'Brazil',
    'Switzerland',         'Sweden',          'Chile', 'United Kingdom',
         'Mexico',        'Germany',      'Argentina',          'Spain',
  'United States',          'Japan',    'South Korea',   'South Africa',
         'Russia',          'Qatar',         'Canada']
Length: 19, dtype: str

In [18]:
continents = {
    "Uruguay": "South America",
    "Italy": "Europe",
    "France": "Europe",
    "Brazil": "South America",
    "Switzerland": "Europe",
    "Sweden": "Europe",
    "Chile": "South America",
    "United Kingdom": "Europe",
    "Mexico": "North America",
    "Germany": "Europe",
    "Argentina": "South America",
    "Spain": "Europe",
    "United States": "North America",
    "Japan": "Asia",
    "South Korea": "Asia",
    "South Africa": "Africa",
    "Russia": "Europe",
    "Qatar": "Asia",
    "Canada": "North America"
}

data["Continent"] = data["Host"].map(continents)
data[["Host", "Continent"]]

,Host,Continent
0,Uruguay,South America
1,Italy,Europe
2,France,Europe
3,Brazil,South America
4,Switzerland,Europe
5,Sweden,Europe
6,Chile,South America
7,United Kingdom,Europe
8,Mexico,North America
9,Germany,Europe


# 7. Export

In [17]:
data.to_csv("../data/processed/hosts.csv", index=False)